### Categorize SASB Metrics (quantitative)

In [90]:
import pandas as pd
import numpy as np
import re

In [91]:
df1 = pd.read_excel(r"SASB Metrics and Data (working).xlsx", sheet_name="Metrics")
df1 = df1[['Topics', 'Quantitative', 'Code','Metrics']]
df1.columns = df1.columns.str.lower()
df1.drop_duplicates(subset=['code'], inplace=True)

# the unique set of SASB metrics that are quantitative
quant = df1[df1['quantitative']==1]
quant.head()

,topics,quantitative,code,metrics
1,Data Privacy,1,SV-AD-220a.2,Percentage of online advertising impressions t...
2,Data Privacy,1,SV-AD-220a.3,Total amount of monetary losses as a result of...
3,Advertising Integrity,1,SV-AD-270a.1,Total amount of monetary losses as a result of...
4,Advertising Integrity,1,SV-AD-270a.2,Percentage of campaigns reviewed for adherence...
5,Advertising Integrity,1,SV-AD-270a.3,Percentage of campaigns that promote alcohol o...


In [92]:
df2 = pd.read_stata("regression2.dta")
df2.columns = df2.columns.str.lower()
df2 = df2[['line_in_master', 'code', 'metrics', 'quantitative', 'reported']]
df2.head()

,line_in_master,code,metrics,quantitative,reported
0,10.0,RT-CH-110a.1,"Gross global Scope 1 emissions, percentage cov...",1.0,1
1,10.0,RT-CH-110a.2,Discussion of long-term and short-term strateg...,0.0,1
2,10.0,RT-CH-120a.1,Air emissions of the following pollutants: (1)...,0.0,1
3,10.0,RT-CH-130a.1,"(1) Total energy consumed, (2) percentage grid...",1.0,1
4,10.0,RT-CH-140a.1,"(1) Total water withdrawn, (2) total water con...",1.0,1


In [93]:
overlap = quant.columns.intersection(df2.columns).difference(['code'])
# Drop overlapping variables from quant (for merging, because the same variable names exist in both dataframes)
df = pd.merge(
    quant.drop(columns=overlap),
    df2,
    on='code',
    how='right',
    indicator=True
)
print(df['_merge'].value_counts())

df.drop(columns=['_merge'], inplace=True)

_merge
both          4173
right_only    1552
left_only        0
Name: count, dtype: int64


Use a dictionary to categorize quant metrics

In [94]:
# metric_topic_keywords = {

#     # Your existing categories
#     'data_breach': [
#         'data breach', 'data breaches',
#         'security breach', 'security breaches'
#     ],

#     'monetary_loss': [
#         'monetary loss', 'monetary losses', 'fraud losses'
#     ],

#     'compliance_incident': [
#         'incidents of non-compliance',
#         'incident of non-compliance',
#         'notices of violation',
#         'notice of violation',
#         'regulatory violation',
#         'regulatory violations',
#         'non-compliance',
#         'noncompliance',
#         'violations of current Good Manufacturing Practices',
#         'pipeline incidents',
#         'number of recalls',
#         'recalls issued',
#         'units recalled',
#         'product recalled',
#         'products recalled',
#         'vehicles recalled'
#     ],

#     'workplace_harm': [
#         'fatality rate',
#         'fatalities',
#         'fatality',
#         'total recordable incident rate',
#         'recordable incident rate',
#         'trir',
#         'lost time incident rate',
#         'near miss',
#         'work stoppages',
#         'gaming staff who work in areas where smoking is allowed',
#         'long-term (chronic) health risks'
#     ],


#     'air_emission': [
#         'greenhouse gas',
#         'ghg emission',
#         'ghg emissions',
#         'scope 1',
#         'scope 2',
#         'scope 3',
#         'carbon emission',
#         'carbon emissions',
#         'air emissions', 'nox', 'sox','particulate matter',
#         'volatile organic compounds', 'voc emissions'
#     ],

#     'water_management': [
#         'water withdrawn',
#         'water withdrawal',
#         'water consumed',
#         'water consumption',
#         'water stress',
#         'water use',
#         'water management',
#         'wastewater',
#         'water loss',
#         'water violations'
#     ],

#     'waste_management': [
#         'hazardous waste',
#         'non-hazardous waste',
#         'nonhazardous waste',
#         'waste generated',
#         'waste recycled',
#         'waste disposed',
#         'waste disposal',
#         'waste management',
#         'landfill',
#         'tailings waste',
#         'tailings impoundment',
#         'processing waste'
#     ],

#     # New categories
#     'energy_management': [
#         'energy consumed',
#         'energy consumption',
#         'grid electricity',
#         'renewable energy',
#         'renewable electricity',
#         'energy intensity',
#         'fuel consumed',
#         'fuel consumption'
#     ],

#     'employee_diversity': [
#         'gender representation',
#         'racial/ethnic',
#         'racial representation',
#         'ethnic representation',
#         'diversity',
#         'female employees',
#         'women in management'
#     ],

#     'product_safety': [
#         'product safety',
#         'safety-related',
#         'safety assessment',
#         'food safety',
#         'product quality',
#         'safety complaints'
#     ],

# }

# quant = quant.copy()

# # Initialize all topic variables to 0
# for topic in metric_topic_keywords:
#     quant[topic] = 0

# # Keep track of whether a metric has already been classified
# classified = pd.Series(False, index=quant.index)

# # Assign categories sequentially
# for topic, keywords in metric_topic_keywords.items():
#     pattern = '|'.join(re.escape(k) for k in keywords)

#     match = (
#         quant['metrics']
#         .fillna('')
#         .astype(str)
#         .str.lower()
#         .str.contains(pattern, regex=True, na=False)
#     )

#     # Only assign observations not previously classified
#     assign = match & ~classified

#     quant.loc[assign, topic] = 1

#     # Mark them as classified
#     classified = classified | assign

# topic_vars = list(metric_topic_keywords.keys())

# quant['other_quant2'] = (
#     quant[topic_vars].sum(axis=1) == 0
# ).astype(int)

Use the "Topics" variable to decide main topics 

In [95]:
# exclude not applicable metrics
df = df[df['reported'] != 2]

df['topics'].value_counts().head(20)

topics
Water Management                                               325
Energy Management                                              204
Greenhouse Gas Emissions                                       161
Workforce Health & Safety                                      116
Labor Practices                                                109
Air Quality                                                    107
Product Safety                                                  93
Operational Safety, Emergency Preparedness & Response           84
Food Safety                                                     82
Product Labeling & Marketing                                    79
Business Ethics                                                 79
Data Security                                                   78
Ecological Impacts                                              70
Reserves Valuation & Capital Expenditures                       65
Supply Chain Management                                

In [96]:
quant = df[df['quantitative'] == 1]
print(len(df), len(quant))   # there are 4,180 quantitative firm-year-metric obs for quantitative metrics
quant['topics'].value_counts().head(20)

5576 4055


topics
Water Management                                               325
Energy Management                                              204
Greenhouse Gas Emissions                                       161
Workforce Health & Safety                                      116
Labor Practices                                                109
Air Quality                                                    107
Product Safety                                                  93
Operational Safety, Emergency Preparedness & Response           84
Food Safety                                                     82
Product Labeling & Marketing                                    79
Business Ethics                                                 79
Data Security                                                   78
Ecological Impacts                                              70
Reserves Valuation & Capital Expenditures                       65
Supply Chain Management                                

In [97]:
# the number of top quant topics
N = 20

quant = quant.copy()

# Identify the N most frequent topics
top_topics = quant['topics'].value_counts().nlargest(N).index

# Create categorical variable
quant['quant_topic'] = quant['topics'].where(
    quant['topics'].isin(top_topics),
    'Other Quant'
)

# Check
quant['quant_topic'].value_counts()

quant_topic
Other Quant                                                    2062
Water Management                                                325
Energy Management                                               204
Greenhouse Gas Emissions                                        161
Workforce Health & Safety                                       116
Labor Practices                                                 109
Air Quality                                                     107
Product Safety                                                   93
Operational Safety, Emergency Preparedness & Response            84
Food Safety                                                      82
Product Labeling & Marketing                                     79
Business Ethics                                                  79
Data Security                                                    78
Ecological Impacts                                               70
Reserves Valuation & Capital Expendi

In [98]:
## Descriptive statistics - report rate for each quant topic

topic_summary = (
    quant.groupby('quant_topic')
      .agg(
          Freq=('reported', 'size'),
          N_Reports=('line_in_master', 'nunique'),
          Report_Rate=('reported', 'mean')
      )
      .reset_index()
)

# Calculate percentage
topic_summary['Percent'] = (
    topic_summary['Freq'] / topic_summary['Freq'].sum() * 100
)

# Sort substantive topics by frequency, but keep Other Quant at the bottom
topic_summary = pd.concat([
    topic_summary[topic_summary['quant_topic'] != 'Other Quant']
        .sort_values('Freq', ascending=False),

    topic_summary[topic_summary['quant_topic'] == 'Other Quant']
]).reset_index(drop=True)

# Calculate cumulative percentage AFTER sorting
topic_summary['Cum_Percent'] = topic_summary['Percent'].cumsum()

# Round
topic_summary['Percent'] = topic_summary['Percent'].round(2)
topic_summary['Cum_Percent'] = topic_summary['Cum_Percent'].round(2)
topic_summary['Report_Rate'] = topic_summary['Report_Rate'].round(3)

topic_summary.to_csv("quant_topics.csv", index=False)

In [99]:
df.columns

Index(['topics', 'code', 'line_in_master', 'metrics', 'quantitative',
       'reported'],
      dtype='object')

In [100]:
cols = [
    'code', 'metrics', 'quant_topic'
]

# Make variable names lowercase
df.columns = df.columns.str.lower()

df.drop_duplicates(subset=['code'], inplace=True)

df.to_stata(
    "quant_topics.dta",
    write_index=False,
    version=118
)

Sensativity test -  combine similar topics

* SASB topic labels can be industry-specific even when the underlying sustainability issue is essentially the same

In [101]:
quant['topics'] = quant['topics'].str.strip()

topics = (
    quant['topics']
    .dropna()
    .drop_duplicates()
    .sort_values()
)

print(topics.to_list())

['Access for Low-Income Patients', 'Access to Coverage', 'Accident & Safety Management', 'Accident Management', 'Advertising Integrity', 'Affordability & Pricing', 'Air Emissions from Refrigeration', 'Air Quality', 'Animal & Feed Sourcing', 'Animal Care & Welfare', 'Antibiotic Use in Animal Production', 'Biodiversity Impacts', 'Business Ethics', 'Business Ethics & Payments Transparency', 'Business Ethics & Transparency', 'Chemical & Safety Hazards of Products', 'Chemicals Management', 'Climate Change Adaptation', 'Climate Change Impacts on Human Health & Infrastructure', 'Climate Impacts of Business Mix', 'Coal Ash Management', 'Community Impacts of New Developments', 'Community Relations', 'Competitive Behavior', 'Competitive Behavior & Open Internet', 'Counterfeit Drugs', 'Critical Incident Risk Management', 'Customer Health & Safety', 'Customer Privacy', 'Customer Privacy & Technology Standards', 'Customer Safety', 'Data Privacy', 'Data Privacy & Advertising Standards', 'Data Privac

In [102]:
topic_map = {  # orignical topic: new topic (a group of similar topics)

    # Accident / operational safety
    'Accident & Safety Management': 'Operational & Accident Safety',
    'Accident Management':'Operational & Accident Safety',
    'Critical Incident Risk Management': 'Operational & Accident Safety',
    'Operational Safety, Emergency Preparedness & Response': 'Operational & Accident Safety',

    # Air emissions / air quality
    'Air Emissions from Refrigeration': 'Air Emissions & Quality',
    'Air Quality': 'Air Emissions & Quality',

    # Business ethics / transparency
    'Business Ethics': 'Business Ethics & Transparency',
    'Business Ethics & Payments Transparency': 'Business Ethics & Transparency',
    'Business Ethics & Transparency':'Business Ethics & Transparency',

    # Customer / data privacy
    'Customer Privacy': 'Data Privacy & Security',
    'Customer Privacy & Technology Standards': 'Data Privacy & Security',
    'Data Privacy': 'Data Privacy & Security',
    'Data Privacy & Advertising Standards': 'Data Privacy & Security',
    'Data Privacy & Freedom of Expression': 'Data Privacy & Security',
    'Data Privacy, Advertising Standards & Freedom of Expression': 'Data Privacy & Security',
    'Data Security': 'Data Privacy & Security',
    'Data Security & Privacy': 'Data Privacy & Security',

    # Ecological impacts
    'Discharge Management & Ecological Impacts': 'Ecological Impacts',
    'Ecological Impacts': 'Ecological Impacts',
    'Ecological Impacts of Project Development': 'Ecological Impacts',
    'Ecosystem Services & Impacts': 'Ecological Impacts',
    'Environmental Impacts of Project Development': 'Ecological Impacts',
    'Land Use & Ecological Impacts': 'Ecological Impacts',

    # Employee/workforce diversity
    'Employee Diversity & Inclusion': 'Workforce Diversity & Inclusion',
    'Workforce Diversity & Engagement': 'Workforce Diversity & Inclusion',
    'Workforce Diversity & Inclusion': 'Workforce Diversity & Inclusion',

    # Employee/workforce safety
    'Employee Health & Safety': 'Workforce Health & Safety',
    'Workforce Health & Safety': 'Workforce Health & Safety',
    'Workforce Safety': 'Workforce Health & Safety',

    # Employee recruitment/development
    'Employee Recruitment, Development & Retention': 'Workforce Recruitment & Development',
    'Employee Recruitment, Inclusion & Performance': 'Workforce Recruitment & Development',
    'Recruiting & Managing a Global & Skilled Workforce': 'Workforce Recruitment & Development',
    'Recruiting & Managing a Global, Diverse & Skilled Workforce': 'Workforce Recruitment & Development',

    # Energy management
    'Energy Management': 'Energy Management',
    'Energy Management in Manufacturing': 'Energy Management',
    'Energy Management in Retail': 'Energy Management',

    # Food safety / product safety
    'Food Safety': 'Product & Food Safety',
    'Product Safety': 'Product & Food Safety',
    'Customer Safety': 'Product & Food Safety',
    'Customer Health & Safety': 'Product & Food Safety',
    'Drug Safety': 'Product & Food Safety',

    # Fuel economy / use-phase emissions
    'Design for Fuel Efficiency': 'Fuel Economy & Use-phase Emissions',
    'Fuel Economy & Emissions in Use-phase': 'Fuel Economy & Use-phase Emissions',
    'Fuel Economy & Use-phase Emissions': 'Fuel Economy & Use-phase Emissions',

    # GHG
    'Greenhouse Gas Emissions': 'Greenhouse Gas Emissions',
    'Greenhouse Gas Emissions & Energy Resource Planning': 'Greenhouse Gas Emissions',

    # Labor
    'Labor Conditions': 'Labor Practices & Conditions',
    'Labor Practices': 'Labor Practices & Conditions',
    'Labor Relations': 'Labor Practices & Conditions',

    # Product lifecycle
    'Packaging Lifecycle Management': 'Product Lifecycle Management',
    'Product Design & Lifecycle Management': 'Product Lifecycle Management',
    'Product End-of-life Management': 'Product Lifecycle Management',
    'Product Lifecycle Environmental Impacts': 'Product Lifecycle Management',
    'Product Lifecycle Management': 'Product Lifecycle Management',

    # Supply chain / sourcing
    'Ingredient Sourcing': 'Supply Chain & Sourcing',
    'Materials Sourcing': 'Supply Chain & Sourcing',
    'Raw Materials Sourcing': 'Supply Chain & Sourcing',
    'Supply Chain Management': 'Supply Chain & Sourcing',
    'Supply Chain Management & Food Sourcing': 'Supply Chain & Sourcing',
    'Wood Supply Chain Management': 'Supply Chain & Sourcing',

    # Waste
    'Food & Packaging Waste Management': 'Waste & Hazardous Materials Management',
    'Food Waste Management': 'Waste & Hazardous Materials Management',
    'Hazardous Materials Management': 'Waste & Hazardous Materials Management',
    'Hazardous Waste Management': 'Waste & Hazardous Materials Management',
    'Management of Leachate & Hazardous Waste': 'Waste & Hazardous Materials Management',
    'Waste & Hazardous Materials Management': 'Waste & Hazardous Materials Management',
    'Waste Management': 'Waste & Hazardous Materials Management',

    # Water
    'Drinking Water Quality': 'Water Management',
    'Effluent Quality Management': 'Water Management',
    'Water Management': 'Water Management',
    'Water Management in Manufacturing': 'Water Management',
    'Water Supply Resilience': 'Water Management'
}


topic_map.update({

    # Marketing / selling practices
    'Product Labeling & Marketing':
        'Marketing & Selling Practices',
    'Ethical Marketing':
        'Marketing & Selling Practices',
    'Marketing Practices':
        'Marketing & Selling Practices',
    'Advertising Integrity':
        'Marketing & Selling Practices',
    'Selling Practices':
        'Marketing & Selling Practices',

    # Affordability / access
    'Affordability & Pricing':
        'Access & Affordability',
    'Energy Affordability':
        'Access & Affordability',
    'Water Affordability & Access':
        'Access & Affordability',
    'Access for Low-Income Patients':
        'Access & Affordability',
    'Access to Coverage':
        'Access & Affordability',

    # Competitive behavior
    'Competitive Behavior':
        'Competitive Behavior',
    'Competitive Behavior & Open Internet':
        'Competitive Behavior',
    'Intellectual Property Protection & Competitive Behavior':
        'Competitive Behavior',

    # Systemic risk / resiliency
    'Systemic Risk Management':
        'Systemic Risk & Resiliency',
    'Managing Systemic Risks from Technology Disruptions':
        'Systemic Risk & Resiliency',

    # Climate resilience / adaptation
    'Climate Change Adaptation':
        'Climate Risk & Resilience',
    'Network Resiliency & Impacts of Climate Change':
        'Climate Risk & Resilience',
    'Climate Change Impacts on Human Health & Infrastructure':
        'Climate Risk & Resilience',
    'Environmental Risk Exposure':
        'Climate Risk & Resilience',
    'Environmental Risk to Mortgaged Properties':
        'Climate Risk & Resilience',

    # Community impacts / relations
    'Community Relations':
        'Community Relations & Impacts',
    'Community Impacts of New Developments':
        'Community Relations & Impacts',

    # Supply-chain environmental/social impacts
    'Environmental & Social Impacts of Ingredient Supply Chain':
        'Environmental & Social Supply Chain Impacts',
    'Environmental Impacts in the Supply Chain':
        'Environmental & Social Supply Chain Impacts',
    'Management of Environmental & Social Impacts in the Supply Chain':
        'Environmental & Social Supply Chain Impacts',
    'Environmental & Social Impacts of Palm Oil Supply':
        'Environmental & Social Supply Chain Impacts',
    'Environmental & Social Impacts of Animal Supply Chain':
        'Environmental & Social Supply Chain Impacts',
    'Sourcing & Environmental Impacts of Feedstock Production':
        'Environmental & Social Supply Chain Impacts',

    # Materials/resource efficiency
    'Materials Efficiency':
        'Materials & Resource Efficiency',
    'Materials Efficiency & Recycling':
        'Materials & Resource Efficiency',
    'Design for Resource Efficiency':
        'Materials & Resource Efficiency',
    'Recycling & Resource Recovery':
        'Materials & Resource Efficiency',
    'Remanufacturing Design & Services':
        'Materials & Resource Efficiency',

    # End-use / product efficiency
    'End-Use Efficiency':
        'Product & End-Use Efficiency',
    'End-Use Efficiency & Demand':
        'Product & End-Use Efficiency',
    'Product Design for Use-phase Efficiency':
        'Product & End-Use Efficiency',
    'Product Efficiency':
        'Product & End-Use Efficiency',

    # Health / nutrition
    'Health & Nutrition':
        'Health & Nutrition',
    'Nutritional Content':
        'Health & Nutrition',
    'Product Health & Nutrition':
        'Health & Nutrition',

    # Healthcare outcomes
    'Quality of Care & Patient Satisfaction':
        'Patient Outcomes & Quality of Care',
    'Patient Health Outcomes':
        'Patient Outcomes & Quality of Care',
    'Improved Outcomes':
        'Patient Outcomes & Quality of Care',

    # Customer information / transparency
    'Transparent Information & Fair Advice for Customers':
        'Customer Information & Transparency',
    'Transparent Information & Management of Conflict of Interest':
        'Customer Information & Transparency',
    'Pricing Integrity & Transparency':
        'Customer Information & Transparency',
    'Pricing & Billing Transparency':
        'Customer Information & Transparency',

    # Chemical management / safety
    'Chemical & Safety Hazards of Products':
        'Chemical Management & Safety',
    'Safety & Environmental Stewardship of Chemicals':
        'Chemical Management & Safety',
    'Management of Chemicals in Products':
        'Chemical Management & Safety',
    'Chemicals Management':
        'Chemical Management & Safety',

    # Indigenous peoples / human rights
    'Rights of Indigenous Peoples':
        'Human & Indigenous Rights',
    'Security, Human Rights & Rights of Indigenous Peoples':
        'Human & Indigenous Rights',

    # ESG integration in financial services
    'Incorporation of Environmental, Social, and Governance Factors in Credit Analysis':
        'ESG Integration in Financial Activities',
    'Incorporation of Environmental, Social, and Governance Factors in Investment Banking & Brokerage Activities':
        'ESG Integration in Financial Activities',
    'Incorporation of Environmental, Social, and Governance Factors in Investment Management':
        'ESG Integration in Financial Activities',
    'Incorporation of Environmental, Social, and Governance Factors in Investment Management & Advisory':
        'ESG Integration in Financial Activities',

    # Environmental footprint
    'Environmental Footprint of Hardware Infrastructure':
        'Environmental Footprint of Operations',
    'Environmental Footprint of Operations':
        'Environmental Footprint of Operations',

    # Product lifecycle / packaging
    'Product Packaging & Distribution':
        'Product Packaging & Lifecycle',
    'Lifecycle Impacts of Buildings & Infrastructure':
        'Product Packaging & Lifecycle',

})

topic_map.update({

    # Infrastructure / operational safety
    'Integrity of Gas Delivery Infrastructure':
        'Infrastructure & Operational Safety',
    'Structural Integrity & Safety':
        'Infrastructure & Operational Safety',

    # Healthcare / drug integrity
    'Counterfeit Drugs':
        'Drug Safety & Supply Chain Integrity',
    'Drug Supply Chain Integrity':
        'Drug Safety & Supply Chain Integrity',

    # Lending practices
    'Lending Practices':
        'Responsible Lending Practices',
    'Discriminatory Lending':
        'Responsible Lending Practices',

    # Animal sourcing / welfare
    'Animal Care & Welfare':
        'Animal Welfare & Sourcing',
    'Animal & Feed Sourcing':
        'Animal Welfare & Sourcing',
    'Antibiotic Use in Animal Production':
        'Animal Welfare & Sourcing',
})

In [103]:
# Construct broad topics
quant['broad_topic'] = (
    quant['topics']
    .map(topic_map)
    .fillna(quant['topics'])
)

# Identify top 20 broad topics
top20 = quant['broad_topic'].value_counts().nlargest(20).index

# Keep top 20; combine everything else into Other_Quant
quant['broad_topic'] = quant['broad_topic'].where(
    quant['broad_topic'].isin(top20),
    'Other_Quant'
)

# Check
quant['broad_topic'].value_counts()

broad_topic
Other_Quant                                    1157
Water Management                                343
Energy Management                               229
Product & Food Safety                           228
Greenhouse Gas Emissions                        193
Data Privacy & Security                         188
Operational & Accident Safety                   168
Workforce Health & Safety                       155
Marketing & Selling Practices                   150
Labor Practices & Conditions                    129
Waste & Hazardous Materials Management          128
Air Emissions & Quality                         122
Business Ethics & Transparency                  119
Product Lifecycle Management                    117
Supply Chain & Sourcing                         110
Ecological Impacts                              110
Access & Affordability                           98
Workforce Recruitment & Development              95
Environmental & Social Supply Chain Impacts      76


In [104]:
## Descriptive statistics - report rate for each quant topic

topic_summary = (
    quant.groupby('broad_topic')
      .agg(
          Freq=('reported', 'size'),
          N_Reports=('line_in_master', 'nunique'),
          Report_Rate=('reported', 'mean')
      )
      .reset_index()
)

# Calculate percentage
topic_summary['Percent'] = (
    topic_summary['Freq'] / topic_summary['Freq'].sum() * 100
)

# Sort substantive topics by frequency, but keep Other Quant at the bottom
topic_summary = pd.concat([
    topic_summary[topic_summary['broad_topic'] != 'Other Quant']
        .sort_values('Freq', ascending=False),

    topic_summary[topic_summary['broad_topic'] == 'Other Quant']
]).reset_index(drop=True)

# Calculate cumulative percentage AFTER sorting
topic_summary['Cum_Percent'] = topic_summary['Percent'].cumsum()

# Round
topic_summary['Percent'] = topic_summary['Percent'].round(2)
topic_summary['Cum_Percent'] = topic_summary['Cum_Percent'].round(2)
topic_summary['Report_Rate'] = topic_summary['Report_Rate'].round(3)

topic_summary.to_csv("quant_topics.csv", index=False)

In [105]:
cols = [
    'code', 'metrics', 'quant_topic'
]

# Make variable names lowercase
quant.columns = quant.columns.str.lower()

quant.drop_duplicates(subset=['code'], inplace=True)


quant.to_stata(
    "quant_topics.dta",
    write_index=False,
    version=118
)